# ATP Tennis ML Project — Data Cleaning & Feature Engineering
## Data Preprocessing, Historical Feature Engineering, and Chronological Train/Test Split

This notebook implements the complete data pipeline for the ATP Tennis match prediction project:
1. **Load and Combine**: Ingests ATP match data from 2018 to 2024 (7 CSV files).
2. **Data Cleaning**: Deduplicates records, standardizes column names, coerces numeric types, and cleans categorical variables.
3. **Historical Feature Engineering**: Calculates pre-match rolling win rates, surface-specific win rates, and head-to-head records without future-data leakage.
4. **Differential ML Features**: Derives ranking differences, age differences, and historical performance differentials for both player perspectives (target 1 and target 0).
5. **Chronological Split**: Splits the dataset into Training (2018–2023) and Testing (2024) sets and exports the handoff datasets.


### 1. Load and Combine the 7 ATP CSV Files (2018–2024)

In [1]:
import pandas as pd
import glob
import os
import numpy as np

# Find all 7 yearly match files
files = sorted(glob.glob("atp_matches_*.csv"))
if not files:
    files = sorted(glob.glob("../atp_matches_*.csv"))
if not files:
    files = sorted(glob.glob("data/atp_matches_*.csv"))

print("Files found:")
for f in files:
    print(" -", f)

if len(files) != 7:
    print(f"WARNING: Expected 7 CSV files (2018-2024), found {len(files)}.")

dfs = []
for f in files:
    df = pd.read_csv(f)
    year = int(os.path.basename(f).split("_")[-1].replace(".csv", ""))
    df["year"] = year
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print("Combined shape:", data.shape)
data.head()

Files found:
 - atp_matches_2018.csv
 - atp_matches_2019.csv
 - atp_matches_2020.csv
 - atp_matches_2021.csv
 - atp_matches_2022.csv
 - atp_matches_2023.csv
 - atp_matches_2024.csv
Combined shape: (18877, 50)


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,year
0,2018-M020,Brisbane,Hard,32,A,20180101,271,105992,NaN,NaN,...,33.0,19.0,14.0,1.0,4.0,47.0,1010.0,52.0,909.0,2018
1,2018-M020,Brisbane,Hard,32,A,20180101,272,111577,NaN,NaN,...,25.0,7.0,9.0,7.0,11.0,54.0,890.0,94.0,593.0,2018
2,2018-M020,Brisbane,Hard,32,A,20180101,273,104797,NaN,NaN,...,37.0,29.0,15.0,10.0,16.0,63.0,809.0,30.0,1391.0,2018
3,2018-M020,Brisbane,Hard,32,A,20180101,275,200282,NaN,WC,...,33.0,17.0,11.0,4.0,6.0,208.0,245.0,44.0,1055.0,2018
4,2018-M020,Brisbane,Hard,32,A,20180101,276,111581,NaN,Q,...,28.0,5.0,9.0,0.0,2.0,175.0,299.0,68.0,755.0,2018


### 2. Clean the Dataset

In [2]:
# Remove exact duplicate rows
data = data.drop_duplicates().reset_index(drop=True)

# Standardize column names
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
)

# Convert important numeric columns
numeric_cols = [
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points",
    "winner_age", "loser_age"
]
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Clean categorical columns
categorical_cols = [
    "surface", "round", "tourney_name",
    "winner_name", "loser_name"
]
for col in categorical_cols:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

# Essential fields needed later
required_cols = [
    "winner_id", "loser_id",
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points",
    "surface"
]

data = data.dropna(subset=required_cols).reset_index(drop=True)

# Age is not essential enough to delete matches; impute missing ages with median
data["winner_age"] = data["winner_age"].fillna(data["winner_age"].median())
data["loser_age"] = data["loser_age"].fillna(data["loser_age"].median())

print("Cleaned shape:", data.shape)
print("\nMissing values in required columns:")
print(data[required_cols].isnull().sum())

Cleaned shape: (18572, 50)

Missing values in required columns:
winner_id             0
loser_id              0
winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
surface               0
dtype: int64


### 3. Preserved Columns & Leakage Prevention Policy

- **Preserved ID & Chronological fields**: `winner_id`, `loser_id`, `winner_rank`, `loser_rank`, `winner_rank_points`, `loser_rank_points`, `winner_age`, `loser_age`, `surface`, `year`, `tourney_date`.
- **Exclusion of post-match statistics**: Match outcomes such as aces, double faults, first-serve percentage, break points, and winners are strictly excluded as model inputs because they are unavailable before match commencement.


### 4. Create Chronological Historical Features
Features are created using only information available **BEFORE** each match to guarantee zero future-data leakage.

In [3]:
history_data = data.copy()

# Sort chronologically by year and tournament date
history_data = history_data.sort_values(
    by=["year", "tourney_date"]
).reset_index(drop=True)

player_stats = {}
h2h = {}
historical_rows = []

def get_player_stats(player_id):
    if player_id not in player_stats:
        player_stats[player_id] = {
            "matches": 0,
            "wins": 0,
            "surface_matches": {},
            "surface_wins": {}
        }
    return player_stats[player_id]

def get_h2h(player1, player2):
    key = tuple(sorted([player1, player2]))
    if key not in h2h:
        h2h[key] = {player1: 0, player2: 0}
    return h2h[key].get(player1, 0)

for _, row in history_data.iterrows():
    winner = row["winner_id"]
    loser = row["loser_id"]
    surface = row["surface"]
    if pd.isna(winner) or pd.isna(loser):
        continue
        
    w = get_player_stats(winner)
    l = get_player_stats(loser)
    
    w_win_rate = w["wins"] / w["matches"] if w["matches"] else 0.5
    l_win_rate = l["wins"] / l["matches"] if l["matches"] else 0.5
    
    w_sm = w["surface_matches"].get(surface, 0)
    w_sw = w["surface_wins"].get(surface, 0)
    l_sm = l["surface_matches"].get(surface, 0)
    l_sw = l["surface_wins"].get(surface, 0)
    
    w_surface_rate = w_sw / w_sm if w_sm else 0.5
    l_surface_rate = l_sw / l_sm if l_sm else 0.5
    
    w_h2h = get_h2h(winner, loser)
    l_h2h = get_h2h(loser, winner)

    # Winner perspective: target = 1
    historical_rows.append({
        "player1": winner,
        "player2": loser,
        "surface": surface,
        "rank1": row["winner_rank"],
        "rank2": row["loser_rank"],
        "points1": row["winner_rank_points"],
        "points2": row["loser_rank_points"],
        "age1": row["winner_age"],
        "age2": row["loser_age"],
        "winrate1": w_win_rate,
        "winrate2": l_win_rate,
        "surfacerate1": w_surface_rate,
        "surfacerate2": l_surface_rate,
        "h2h1": w_h2h,
        "h2h2": l_h2h,
        "target": 1,
        "year": row["year"]
    })

    # Loser perspective: target = 0
    historical_rows.append({
        "player1": loser,
        "player2": winner,
        "surface": surface,
        "rank1": row["loser_rank"],
        "rank2": row["winner_rank"],
        "points1": row["loser_rank_points"],
        "points2": row["winner_rank_points"],
        "age1": row["loser_age"],
        "age2": row["winner_age"],
        "winrate1": l_win_rate,
        "winrate2": w_win_rate,
        "surfacerate1": l_surface_rate,
        "surfacerate2": w_surface_rate,
        "h2h1": l_h2h,
        "h2h2": w_h2h,
        "target": 0,
        "year": row["year"]
    })

    # Update history AFTER this match
    w["matches"] += 1
    w["wins"] += 1
    l["matches"] += 1
    w["surface_matches"][surface] = w["surface_matches"].get(surface, 0) + 1
    w["surface_wins"][surface] = w["surface_wins"].get(surface, 0) + 1
    l["surface_matches"][surface] = l["surface_matches"].get(surface, 0) + 1
    
    key = tuple(sorted([winner, loser]))
    if key not in h2h:
        h2h[key] = {winner: 0, loser: 0}
    h2h[key][winner] = h2h[key].get(winner, 0) + 1

historical_data = pd.DataFrame(historical_rows)
print("Historical dataset shape:", historical_data.shape)
historical_data.head()

Historical dataset shape: (37144, 17)


,player1,player2,surface,rank1,rank2,points1,points2,age1,age2,winrate1,winrate2,surfacerate1,surfacerate2,h2h1,h2h2,target,year
0,105992,104919,Hard,47.0,52.0,1010.0,909.0,25.6,30.6,0.5,0.5,0.5,0.5,0,0,1,2018
1,104919,105992,Hard,52.0,47.0,909.0,1010.0,30.6,25.6,0.5,0.5,0.5,0.5,0,0,0,2018
2,111577,111442,Hard,54.0,94.0,890.0,593.0,21.2,23.7,0.5,0.5,0.5,0.5,0,0,1,2018
3,111442,111577,Hard,94.0,54.0,593.0,890.0,23.7,21.2,0.5,0.5,0.5,0.5,0,0,0,2018
4,104797,106000,Hard,63.0,30.0,809.0,1391.0,31.3,25.6,0.5,0.5,0.5,0.5,0,0,1,2018


### 5. Convert Historical Information into Final ML Features

In [4]:
improved_data = pd.DataFrame()

improved_data["rank_diff"] = (
    historical_data["rank1"] - historical_data["rank2"]
)
improved_data["rank_points_diff"] = (
    historical_data["points1"] - historical_data["points2"]
)
improved_data["age_diff"] = (
    historical_data["age1"] - historical_data["age2"]
)
improved_data["win_rate_diff"] = (
    historical_data["winrate1"] - historical_data["winrate2"]
)
improved_data["surface_rate_diff"] = (
    historical_data["surfacerate1"] - historical_data["surfacerate2"]
)
improved_data["h2h_diff"] = (
    historical_data["h2h1"] - historical_data["h2h2"]
)
improved_data["surface"] = historical_data["surface"]
improved_data["year"] = historical_data["year"]
improved_data["target"] = historical_data["target"]

numeric_features = [
    "rank_diff",
    "rank_points_diff",
    "age_diff",
    "win_rate_diff",
    "surface_rate_diff",
    "h2h_diff"
]

for col in numeric_features:
    improved_data[col] = improved_data[col].fillna(
        improved_data[col].median()
    )

print("Final feature dataset shape:", improved_data.shape)
print("\nMissing values count:")
print(improved_data.isnull().sum())
improved_data.head()

Final feature dataset shape: (37144, 9)

Missing values count:
rank_diff            0
rank_points_diff     0
age_diff             0
win_rate_diff        0
surface_rate_diff    0
h2h_diff             0
surface              0
year                 0
target               0
dtype: int64


,rank_diff,rank_points_diff,age_diff,win_rate_diff,surface_rate_diff,h2h_diff,surface,year,target
0,-5.0,101.0,-5.0,0.0,0.0,0,Hard,2018,1
1,5.0,-101.0,5.0,0.0,0.0,0,Hard,2018,0
2,-40.0,297.0,-2.5,0.0,0.0,0,Hard,2018,1
3,40.0,-297.0,2.5,0.0,0.0,0,Hard,2018,0
4,33.0,-582.0,5.7,0.0,0.0,0,Hard,2018,1


### 6. Chronological Train / Test Split (Handoff Datasets)
- **Training Set**: 2018 – 2023 matches
- **Testing Set**: 2024 matches

In [5]:
# 2018-2023 = training, 2024 = testing
train_data = improved_data[improved_data["year"] < 2024].copy()
test_data = improved_data[improved_data["year"] == 2024].copy()

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))
print("Training years:", sorted(train_data["year"].unique()))
print("Testing years:", sorted(test_data["year"].unique()))

# Export data files to data/ directory and root for teammate handoff
os.makedirs("data", exist_ok=True)
os.makedirs("../data", exist_ok=True)

# Save to data/
train_data.to_csv("data/train_data.csv", index=False)
test_data.to_csv("data/test_data.csv", index=False)
data.to_csv("data/cleaned_atp_data.csv", index=False)

# Also save directly to parent or root directory if working in notebooks subfolder
try:
    train_data.to_csv("../data/train_data.csv", index=False)
    test_data.to_csv("../data/test_data.csv", index=False)
    data.to_csv("../data/cleaned_atp_data.csv", index=False)
except Exception:
    pass

print("Saved successfully:")
print(" - data/train_data.csv")
print(" - data/test_data.csv")
print(" - data/cleaned_atp_data.csv")

Training samples: 31084
Testing samples: 6060
Training years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Testing years: [np.int64(2024)]


Saved successfully:
 - data/train_data.csv
 - data/test_data.csv
 - data/cleaned_atp_data.csv


### 7. Handoff Verification & Sanity Checks

In [6]:
print("=== HANDOFF SUMMARY ===")
print(f"Total Matches Processed: {len(data):,}")
print(f"Historical Engineered Rows: {len(improved_data):,}")
print(f"Train Rows (2018-2023): {len(train_data):,}")
print(f"Test Rows (2024): {len(test_data):,}")
print(f"Target Distribution in Train: {dict(train_data['target'].value_counts())}")
print(f"Target Distribution in Test: {dict(test_data['target'].value_counts())}")
print(f"Surface Distribution in Train: {dict(train_data['surface'].value_counts())}")
print(f"Surface Distribution in Test: {dict(test_data['surface'].value_counts())}")


=== HANDOFF SUMMARY ===
Total Matches Processed: 18,572
Historical Engineered Rows: 37,144
Train Rows (2018-2023): 31,084
Test Rows (2024): 6,060
Target Distribution in Train: {1: np.int64(15542), 0: np.int64(15542)}
Target Distribution in Test: {1: np.int64(3030), 0: np.int64(3030)}
Surface Distribution in Train: {'Hard': np.int64(18600), 'Clay': np.int64(9292), 'Grass': np.int64(3192)}
Surface Distribution in Test: {'Hard': np.int64(3482), 'Clay': np.int64(1932), 'Grass': np.int64(646)}
